# 02 — Prepare Labels
Merge ECOSoundSet and InsectSet459, balance classes, and produce the `train.csv` / `val.csv` / `test.csv` splits that OpenSoundscape expects.

**Kernel:** `Python (orthoptera-training)`

In [ ]:
import pandas as pd
import numpy as np
import os

ECO_ROOT    = "../../datasets/ecosoundset"
INSECT_ROOT = "../../datasets/insectset459"
OUT_DIR     = "../../training/data"
os.makedirs(OUT_DIR, exist_ok=True)

# Canonical species names as they appear in ECOSoundSet (trinomial).
# InsectSet459 uses underscore format — mapped below.
# Meconema thalassinum is absent from both datasets.
UK_SPECIES = [
    "Chorthippus brunneus brunneus",          # Field Grasshopper
    "Pseudochorthippus parallelus parallelus", # Meadow Grasshopper
    "Omocestus viridulus",                    # Common Green Grasshopper
    "Tettigonia viridissima",                 # Great Green Bush-cricket
    "Roeseliana roeselii",                    # Roesel's Bush-cricket
    "Pholidoptera griseoaptera",              # Dark Bush-cricket
    "Leptophyes punctatissima",               # Speckled Bush-cricket
    "Gryllus campestris",                     # Field Cricket
]

# InsectSet459 species_name uses underscores and may be binomial.
# Map eco canonical name → insect prefix for matching.
ECO_TO_INSECT = {
    "Chorthippus brunneus brunneus":          "Chorthippus_brunneus",
    "Pseudochorthippus parallelus parallelus": "Pseudochorthippus_parallelus",
    "Omocestus viridulus":                    "Omocestus_viridulus",
    "Tettigonia viridissima":                 "Tettigonia_viridissima",
    "Roeseliana roeselii":                    "Roeseliana_roeselii",
    "Pholidoptera griseoaptera":              "Pholidoptera_griseoaptera",
    "Leptophyes punctatissima":               "Leptophyes_punctatissima",
    "Gryllus campestris":                     "Gryllus_campestris",
}


In [ ]:
# ── Load ECOSoundSet and split by subset column ───────────────────────────────
all_annot = pd.read_csv(os.path.join(ECO_ROOT, "annotated_audio_segments.csv"))
orth = all_annot[
    (all_annot["label_category"] == "Orthoptera") &
    (all_annot["label"].isin(UK_SPECIES))
].copy()
orth["filepath"] = orth["audio_segment_file_name"].apply(
    lambda f: os.path.abspath(os.path.join(ECO_ROOT, f))
)
orth.rename(columns={
    "label": "species",
    "audio_segment_initial_time": "t_min",
    "audio_segment_final_time":   "t_max",
}, inplace=True)

eco_train = orth[orth["subset"] == "train"]
eco_val   = orth[orth["subset"] == "val"]
eco_test  = orth[orth["subset"] == "test"]
print(f"ECOSoundSet — train: {len(eco_train)}  val: {len(eco_val)}  test: {len(eco_test)}")
eco_train["species"].value_counts()


In [ ]:
# ── Supplement with InsectSet459 (train split only) ──────────────────────────
# InsectSet459 has no test split — we use it only to top up training data.
insect_csv = os.path.join(INSECT_ROOT, "InsectSet459_Train_Val_Annotation.csv")
insect_all = pd.read_csv(insect_csv)

# Build reverse map: insect prefix → eco canonical name
insect_prefixes = {v: k for k, v in ECO_TO_INSECT.items()}

def map_insect_species(name):
    for prefix, canonical in insect_prefixes.items():
        if name.startswith(prefix):
            return canonical
    return None

insect_all["species"] = insect_all["species_name"].apply(map_insect_species)
insect_all = insect_all[insect_all["species"].notna()].copy()

# Audio lives in Train/ or Validation/ subdirectory
def insect_audio_path(row):
    subdir = "Train" if row["subset"] == "Train" else "Validation"
    return os.path.abspath(os.path.join(INSECT_ROOT, subdir, row["file_name"]))

insect_all["filepath"] = insect_all.apply(insect_audio_path, axis=1)
insect_all["t_min"] = 0.0
insect_all["t_max"] = 4.0  # InsectSet459 clips are typically ~4 s

insect_train = insect_all[insect_all["subset"] == "Train"]
print(f"InsectSet459 — train: {len(insect_train)}")
insect_train["species"].value_counts()


In [ ]:
# ── Merge and write OpenSoundscape one-hot CSVs ──────────────────────────────
# OSS expects index (file, start_time, end_time) + one-hot species columns.

def to_oss_format(df, species_list):
    rows = []
    for _, row in df.iterrows():
        entry = {
            "file":       row["filepath"],
            "start_time": row.get("t_min", 0.0),
            "end_time":   row.get("t_max", 4.0),
        }
        for sp in species_list:
            entry[sp] = 1 if row["species"] == sp else 0
        rows.append(entry)
    return pd.DataFrame(rows).set_index(["file", "start_time", "end_time"])

# Combine ECO + InsectSet459 for training; use ECO only for val/test
combined_train = pd.concat([eco_train, insect_train], ignore_index=True)

train_oss = to_oss_format(combined_train, UK_SPECIES)
val_oss   = to_oss_format(eco_val,        UK_SPECIES)
test_oss  = to_oss_format(eco_test,       UK_SPECIES)

train_oss.to_csv(os.path.join(OUT_DIR, "train.csv"))
val_oss.to_csv(os.path.join(OUT_DIR,   "val.csv"))
test_oss.to_csv(os.path.join(OUT_DIR,  "test.csv"))

print(f"Saved to {OUT_DIR}/")
print(f"Train: {len(train_oss)}  Val: {len(val_oss)}  Test: {len(test_oss)}")
print("\nPer-species train counts:")
print(train_oss.sum().sort_values(ascending=False).to_string())
